In [42]:
import numpy as np

In [43]:
import pandas as pd

In [44]:
dataset = pd.read_csv("Social_Network_Ads.csv")

In [45]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [46]:
dataset= pd.get_dummies(dataset, dtype=int, drop_first=True)

In [47]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [48]:
dataset = dataset.drop("User ID", axis=1)

In [49]:
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [50]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [51]:
independent = dataset[["Age","EstimatedSalary","Gender_Male"]]

In [52]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [53]:
dependent = dataset[["Purchased"]]

In [54]:
dependent

,Purchased
0,0
1,0
2,0
3,0
4,0
...,...
395,1
396,1
397,1
398,0


In [55]:
#dataset("Purchased").value_counts()

In [56]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(independent, dependent, test_size =0.30, random_state=0)

In [57]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)


In [58]:
from sklearn.linear_model import LogisticRegression
#Classifier = LogisticRegression(random_state = 0)
#Classifier.fit(x_train, np.ravel(y_train))
#y_pred = Classifier.predict(x_test)
from sklearn.model_selection import GridSearchCV
param_grid = {'solver':['newton-cg','lbfgs','liblinear','saga'], 'penalty':['l2']}
grid = GridSearchCV(LogisticRegression(),param_grid,refit = True, verbose = 3, n_jobs=-1, scoring = 'f1_weighted')
grid.fit(x_train, np.ravel(y_train))
re=grid.cv_results_
grid_predictions = grid.predict(x_test)


Fitting 5 folds for each of 4 candidates, totalling 20 fits


In [59]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)
print(cm)

[[74  5]
 [ 8 33]]


In [60]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)
print(clf_report)

              precision    recall  f1-score   support

           0       0.90      0.94      0.92        79
           1       0.87      0.80      0.84        41

    accuracy                           0.89       120
   macro avg       0.89      0.87      0.88       120
weighted avg       0.89      0.89      0.89       120



In [61]:
#from sklearn.metrics import f1_score
#f1_macro=f1_score(y_test, grid_predictions, average = 'weighted')
#print("The f1_macro value for best parameter{}:".format(grid.best_params_),f1_macro)
print("The best parameter set for this model:\n", format(grid.best_params_))
print("The Confusion Matrix:\n",cm)
print("The report:\n", clf_report)
from sklearn.metrics import roc_auc_score
roc_score = roc_auc_score(y_test, grid.predict_proba(x_test)[:,1])
print("The ROC_AUC score for this model is:\n", roc_score)

The best parameter set for this model:
 {'penalty': 'l2', 'solver': 'newton-cg'}
The Confusion Matrix:
 [[74  5]
 [ 8 33]]
The report:
               precision    recall  f1-score   support

           0       0.90      0.94      0.92        79
           1       0.87      0.80      0.84        41

    accuracy                           0.89       120
   macro avg       0.89      0.87      0.88       120
weighted avg       0.89      0.89      0.89       120

The ROC_AUC score for this model is:
 0.9481321395492436


In [62]:
table = pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.007689,0.001487,0.006175,0.000855,l2,newton-cg,"{'penalty': 'l2', 'solver': 'newton-cg'}",0.835985,0.802399,0.644599,0.927778,0.890114,0.820175,0.097839,1
1,0.008510,0.002713,0.005708,0.001776,l2,lbfgs,"{'penalty': 'l2', 'solver': 'lbfgs'}",0.835985,0.802399,0.644599,0.927778,0.890114,0.820175,0.097839,1
2,0.003413,0.000760,0.006660,0.000693,l2,liblinear,"{'penalty': 'l2', 'solver': 'liblinear'}",0.835985,0.802399,0.644599,0.927778,0.890114,0.820175,0.097839,1
3,0.003379,0.000644,0.005691,0.000980,l2,saga,"{'penalty': 'l2', 'solver': 'saga'}",0.835985,0.802399,0.644599,0.927778,0.890114,0.820175,0.097839,1


In [63]:
Age_input = float(input("Age="))
Est_salary_input = float(input("Estimated Salary="))
Gender_male_input = int(input("Gender Male="))

Age= 31
Estimated Salary= 9000
Gender Male= 0


In [64]:
Future_predictions = grid.predict([[Age_input, Est_salary_input, Gender_male_input]])
print("Future Predictions:\n", Future_predictions)

Future Predictions:
 [1]
